In [1]:
from google.cloud import bigquery
import pandas as pd

PROJECT_ID = "pacey32-agency"

SCHEDULE_TABLE = "nhl-pacey32-github.NHL_Views.Schedule"
TEAM_TABLE = "pacey32-agency.Team.TeamList"
CITY_TABLE = "pacey32-agency.City.CityReference"

bq = bigquery.Client(project=PROJECT_ID)

In [2]:
# Latest season
latest_season = bq.query(f"""
SELECT MAX(season) AS season
FROM `{SCHEDULE_TABLE}`
""").result().to_dataframe().iloc[0]["season"]

print(f"Latest season: {latest_season}")

# Schedule
schedule = bq.query(f"""
SELECT *
FROM `{SCHEDULE_TABLE}`
WHERE season = {latest_season}
  AND game_type = 2
  AND game_schedule_state = 'OK'
""").result().to_dataframe()

# Team list
team = bq.query(f"""
SELECT *
FROM `{TEAM_TABLE}`
""").result().to_dataframe()

# City reference
city = bq.query(f"""
SELECT *
FROM `{CITY_TABLE}`
""").result().to_dataframe()

print(len(schedule), "games")
print(len(team), "teams")
print(len(city), "cities")

Latest season: 20252026
1311 games
32 teams
32 cities


In [3]:
team_locations = (
    team.merge(
        city,
        how="left",
        left_on="venueLocation",
        right_on="city_name"
    )
)

team_locations = team_locations.rename(columns={
    "id": "team_id"
})

team_locations = team_locations[
    [
        "team_id",
        "fullName",
        "tricode",
        "venue",
        "venueLocation",
        "state_province",
        "country",
        "latitude",
        "longitude",
        "geography"
    ]
]

print(team_locations.shape)

display(team_locations.sort_values("team_id"))

assert len(team_locations) == 32
assert 68 in team_locations.team_id.values

(32, 10)


,team_id,fullName,tricode,venue,venueLocation,state_province,country,latitude,longitude,geography
0,1,New Jersey Devils,NJD,Prudential Center,Newark,New Jersey,United States,40.73566,-74.17237,POINT(-74.17237 40.73566)
1,2,New York Islanders,NYI,UBS Arena,Elmont,New York,United States,40.70094,-73.71291,POINT(-73.71291 40.70094)
2,3,New York Rangers,NYR,Madison Square Garden,New York,New York,United States,40.71427,-74.00597,POINT(-74.00597 40.71427)
3,4,Philadelphia Flyers,PHI,Xfinity Mobile Arena,Philadelphia,Pennsylvania,United States,39.95238,-75.16362,POINT(-75.16362 39.95238)
4,5,Pittsburgh Penguins,PIT,PPG Paints Arena,Pittsburgh,Pennsylvania,United States,40.44062,-79.99589,POINT(-79.99589 40.44062)
5,6,Boston Bruins,BOS,TD Garden,Boston,Massachusetts,United States,42.35843,-71.05977,POINT(-71.05977 42.35843)
6,7,Buffalo Sabres,BUF,KeyBank Center,Buffalo,New York,United States,42.88645,-78.87837,POINT(-78.87837 42.88645)
7,8,Montréal Canadiens,MTL,Centre Bell,Montreal,Quebec,Canada,45.50884,-73.58781,POINT(-73.58781 45.50884)
8,9,Ottawa Senators,OTT,Canadian Tire Centre,Ottawa,Ontario,Canada,45.41117,-75.69812,POINT(-75.69812 45.41117)
9,10,Toronto Maple Leafs,TOR,Scotiabank Arena,Toronto,Ontario,Canada,43.70643,-79.39864,POINT(-79.39864 43.70643)


In [4]:
home = team_locations.add_prefix("home_")

home_games = (
    schedule
    .merge(
        home,
        how="left",
        left_on="home_team_id",
        right_on="home_team_id"
    )
)

home_games = pd.DataFrame({
    "season": home_games["season"],
    "game_id": home_games["game_id"],
    "game_date": pd.to_datetime(home_games["game_date"]).dt.date,
    "game_datetime": pd.to_datetime(home_games["start_time_utc"]),
    "game_state": home_games["game_state"],
    "game_schedule_state": home_games["game_schedule_state"],
    "game_type": home_games["game_type"],
    "neutral_site": home_games["neutral_site"],
    "is_home": True,
    "is_away": False,
    "team_id": home_games["home_team_id"],
    "team_abbrev": home_games["home_team_abbrev"],
    "team_name": home_games["home_team_name"],
    "opponent_team_id": home_games["away_team_id"],
    "opponent_team_abbrev": home_games["away_team_abbrev"],
    "opponent_team_name": home_games["away_team_name"],
    "game_city": home_games["home_venueLocation"],
    "game_state_province": home_games["home_state_province"],
    "game_country": home_games["home_country"],
    "game_latitude": home_games["home_latitude"],
    "game_longitude": home_games["home_longitude"],
    "game_geography": home_games["home_geography"],
    "team_home_city": home_games["home_venueLocation"],
    "team_home_state_province": home_games["home_state_province"],
    "team_home_country": home_games["home_country"],
    "team_home_latitude": home_games["home_latitude"],
    "team_home_longitude": home_games["home_longitude"],
    "team_home_geography": home_games["home_geography"],
})

In [5]:
home = team_locations.add_prefix("home_")
away = team_locations.add_prefix("away_")

away_games = (
    schedule
    .merge(
        home,
        how="left",
        left_on="home_team_id",
        right_on="home_team_id"
    )
    .merge(
        away,
        how="left",
        left_on="away_team_id",
        right_on="away_team_id"
    )
)

away_games = pd.DataFrame({
    "season": away_games["season"],
    "game_id": away_games["game_id"],
    "game_date": pd.to_datetime(away_games["game_date"]).dt.date,
    "game_datetime": pd.to_datetime(away_games["start_time_utc"]),
    "game_state": away_games["game_state"],
    "game_schedule_state": away_games["game_schedule_state"],
    "game_type": away_games["game_type"],
    "neutral_site": away_games["neutral_site"],
    "is_home": False,
    "is_away": True,
    "team_id": away_games["away_team_id"],
    "team_abbrev": away_games["away_team_abbrev"],
    "team_name": away_games["away_team_name"],
    "opponent_team_id": away_games["home_team_id"],
    "opponent_team_abbrev": away_games["home_team_abbrev"],
    "opponent_team_name": away_games["home_team_name"],
    "game_city": away_games["home_venueLocation"],
    "game_state_province": away_games["home_state_province"],
    "game_country": away_games["home_country"],
    "game_latitude": away_games["home_latitude"],
    "game_longitude": away_games["home_longitude"],
    "game_geography": away_games["home_geography"],
    "team_home_city": away_games["away_venueLocation"],
    "team_home_state_province": away_games["away_state_province"],
    "team_home_country": away_games["away_country"],
    "team_home_latitude": away_games["away_latitude"],
    "team_home_longitude": away_games["away_longitude"],
    "team_home_geography": away_games["away_geography"],
})

In [6]:
travel = pd.concat(
    [home_games, away_games],
    ignore_index=True
)

travel = travel.sort_values(
    ["season", "team_id", "game_date", "game_datetime", "game_id"]
)

travel["team_game_number"] = (
    travel
    .groupby(["season", "team_id"])
    .cumcount()
    + 1
)

display(travel.head())

print(len(travel))

,season,game_id,game_date,game_datetime,game_state,game_schedule_state,game_type,neutral_site,is_home,is_away,...,game_latitude,game_longitude,game_geography,team_home_city,team_home_state_province,team_home_country,team_home_latitude,team_home_longitude,team_home_geography,team_game_number
1324,20252026,2025020014,2025-10-09,2025-10-09 23:30:00+00:00,OFF,OK,2,False,False,True,...,35.77210,-78.63861,POINT(-78.63861 35.7721),Newark,New Jersey,United States,40.73566,-74.17237,POINT(-74.17237 40.73566),1
1336,20252026,2025020026,2025-10-11,2025-10-11 23:00:00+00:00,OFF,OK,2,False,False,True,...,27.94752,-82.45843,POINT(-82.45843 27.94752),Newark,New Jersey,United States,40.73566,-74.17237,POINT(-74.17237 40.73566),2
1355,20252026,2025020045,2025-10-13,2025-10-13 23:00:00+00:00,OFF,OK,2,False,False,True,...,39.96118,-82.99879,POINT(-82.99879 39.96118),Newark,New Jersey,United States,40.73566,-74.17237,POINT(-74.17237 40.73566),3
63,20252026,2025020064,2025-10-16,2025-10-16 23:00:00+00:00,OFF,OK,2,False,True,False,...,40.73566,-74.17237,POINT(-74.17237 40.73566),Newark,New Jersey,United States,40.73566,-74.17237,POINT(-74.17237 40.73566),4
77,20252026,2025020078,2025-10-18,2025-10-18 19:30:00+00:00,OFF,OK,2,False,True,False,...,40.73566,-74.17237,POINT(-74.17237 40.73566),Newark,New Jersey,United States,40.73566,-74.17237,POINT(-74.17237 40.73566),5


2622


In [7]:
display(
    travel[
        (travel["team_abbrev"] == "BOS")
        &
        (travel["opponent_team_abbrev"] == "UTA")
    ]
)

,season,game_id,game_date,game_datetime,game_state,game_schedule_state,game_type,neutral_site,is_home,is_away,...,game_latitude,game_longitude,game_geography,team_home_city,team_home_state_province,team_home_country,team_home_latitude,team_home_longitude,team_home_geography,team_game_number
1402,20252026,2025020092,2025-10-19,2025-10-19 23:00:00+00:00,OFF,OK,2,False,False,True,...,40.76078,-111.89105,POINT(-111.89105 40.76078),Boston,Massachusetts,United States,42.35843,-71.05977,POINT(-71.05977 42.35843),7
518,20252026,2025020519,2025-12-16,2025-12-17 00:00:00+00:00,FUT,OK,2,False,True,False,...,42.35843,-71.05977,POINT(-71.05977 42.35843),Boston,Massachusetts,United States,42.35843,-71.05977,POINT(-71.05977 42.35843),34
